# Seq2Seq 모델 Q&A Chatbot 구현

1. QnA 데이터셋을 찾아서 처리해서 준비한다. (전처리 전반)
2. Encoder, Decoder, Seq2Seq(Encoder + Decoder) 모델을 만든다.
3. 1에서 준비한 데이터로 2에서 만든 모델을 학습시킨다.
4. Chatbot을 만든다. (모델 추론 + while문)

### 1. 데이터셋

In [1]:
# 만약 쓸 데이터셋이 없다면 다음을 사용해보세요...

import pandas as pd

df = pd.read_csv('https://raw.githubusercontent.com/songys/Chatbot_data/refs/heads/master/ChatbotData.csv')
df = df[['Q', 'A']]
df

,Q,A
0,12시 땡!,하루가 또 가네요.
1,1지망 학교 떨어졌어,위로해 드립니다.
2,3박4일 놀러가고 싶다,여행은 언제나 좋죠.
3,3박4일 정도 놀러가고 싶다,여행은 언제나 좋죠.
4,PPL 심하네,눈살이 찌푸려지죠.
...,...,...
11818,훔쳐보는 것도 눈치 보임.,티가 나니까 눈치가 보이는 거죠!
11819,훔쳐보는 것도 눈치 보임.,훔쳐보는 거 티나나봐요.
11820,흑기사 해주는 짝남.,설렜겠어요.
11821,힘든 연애 좋은 연애라는게 무슨 차이일까?,잘 헤어질 수 있는 사이 여부인 거 같아요.


---

### 1.5. 전역변수 설정

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [3]:
BATCH_SIZE = 64
MAX_VOCAB_SIZE = 10000
EMBEDDING_DIM = 100
LATENT_DIM = 512

---

### 2. 데이터 학습에 알맞게 변경하기

In [4]:
q_inputs = []
ans_inputs = []
ans_targets = []

for idx in df.index:
  q = df.loc[idx,'Q']
  ans = df.loc[idx,'A']

  ans_input = '<sos> ' + ans
  ans_target = ans + ' <eos>'

  q_inputs.append(q)
  ans_inputs.append(ans_input)
  ans_targets.append(ans_target)

In [5]:
# 확인했으니 지워도 됨니다

print(len(q_inputs), len(ans_inputs), len(ans_targets))
print(q_inputs[5000])
print(ans_inputs[5000])
print(ans_targets[5000])

11823 11823 11823
학원폭력 짜증나
<sos> 학교 폭력은 범죄에요.
학교 폭력은 범죄에요. <eos>


In [ ]:
kor_tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE, filters='')
kor_tokenizer.fit_on_texts(q_inputs + ans_inputs + ans_targets)

q_seqs = kor_tokenizer.texts_to_sequences(q_inputs)
q_num_words = len(eng_tokenizer.word_index) + 1
q_max_len = max(len(s) for s in eng_seqs)

ans_input_seqs = kor_tokenizer.texts_to_sequences(ans_inputs)
ans_target_seqs = kor_tokenizer.texts_to_sequences(ans_targets)

ans_num_words = len(kor_tokenizer.word_index) + 1
ans_max_len = max(len(s) for s in ans_input_seqs)

print(f'{ans_num_words = }')
print(f'{ans_max_len = }')

ans_num_words = 21800
ans_max_len = 22


In [ ]:
encoder_inputs = pad_sequences(q_seqs, maxlen=eng_max_len, padding='pre')
decoder_inputs = pad_sequences(ans_input_seqs, maxlen=kor_max_len, padding='post')
decoder_targets = pad_sequences(ans_target_seqs, maxlen=kor_max_len, padding='post')

print(encoder_inputs.shape)
print(decoder_inputs.shape)
print(decoder_targets.shape)

print(encoder_inputs[1000])
print([eng_tokenizer.index_word[s] for s in encoder_inputs[1000] if s != 0])
print(decoder_inputs[1000])
print([kor_tokenizer.index_word[s] for s in decoder_inputs[1000] if s != 0])
print(decoder_targets[1000])
print([kor_tokenizer.index_word[s] for s in decoder_targets[1000] if s != 0])

NameError: name 'eng_seqs' is not defined

---

### 3. Seq2Seq 모델 만들기

In [ ]:
import numpy as np

def make_embedding_matrix(num_words, embedding_dim=100, tokenizer=kor_tokenizer):
  embedding_dict = {}

  with open('./glove.6B.100d.txt', 'r', encoding='utf-8') as f:
    for line in f:
      word, *coef = line.split()
      coef = np.array(coef, dtype=float)
      embedding_dict[word] = coef

  embedding_matrix = np.zeros((num_words, embedding_dim))

  for word, index in tokenizer.word_index.items():
    vec = embedding_dict.get(word)
    if vec is not None:
      embedding_matrix[index] = vec
    else:
      embedding_matrix[index] = np.random.rand(embedding_dim)

  return embedding_matrix

embedding_matrix = make_embedding_matrix(q_num_words, EMBEDDING_DIM, eng_tokenizer)
print(embedding_matrix.shape)

##### 인코더 만들기

In [ ]:
class Encoder(nn.Module):
  def __init__(self, vocab_size, embedding_dim, latent_dim, embedding_matrix=None):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
    if embedding_matrix is not None:
      self.embedding.weight.data.copy_(torch.from_numpy(embedding_matrix))
    self.lstm = nn.LSTM(embedding_dim, latent_dim, batch_first=True)

  def forward(self, X):
    X = self.embedding(X)
    output, (h_s, c_s) = self.lstm(X)
    return h_s, c_s

##### 디코더 만들기

In [ ]:
class Decoder(nn.Module):
  def __init__(self, vocab_size, embedding_dim, latent_dim):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
    self.lstm = nn.LSTM(embedding_dim, latent_dim, batch_first=True)
    self.fc = nn.Linear(latent_dim, vocab_size)

  def forward(self, X, hidden, cell):
    X = self.embedding(X)
    output, (h_s, c_s) = self.lstm(X, (hidden, cell))
    logits = self.fc(output)
    return logits, h_s, c_s

##### Seq2Seq 완성하기

In [ ]:
class Seq2Seq(nn.Module):
  def __init__(self, encoder, decoder):
    super().__init__()
    self.encoder = encoder
    self.decoder = decoder

  def forward(self, source, target):
    h_s, c_s = self.encoder(source)
    output, h_s, c_s = self.decoder(target, h_s, c_s)
    return output

---

### 4. Seq2Seq 모델 학습

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

encoder = Encoder(q_num_words, EMBEDDING_DIM, LATENT_DIM, embedding_matrix)
decoder = Decoder(ans_num_words, EMBEDDING_DIM, LATENT_DIM)
model = Seq2Seq(encoder, decoder).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = optim.AdamW(model.parameters(), lr=0.001)

epochs = 100

train_losses, train_accs, val_losses, val_accs = [], [], [], []

for epoch in range(epochs):
  model.train()
  train_loss, train_correct, train_tokens = 0, 0, 0

  for enc_inputs, dec_inputs, dec_targets in train_loader:
    enc_inputs = enc_inputs.to(device)
    dec_inputs = dec_inputs.to(device)
    dec_targets = dec_targets.to(device)

    optimizer.zero_grad()

    # teacher forcing
    output = model(enc_inputs, dec_inputs)
    output = output.view(-1, output.size(-1))
    dec_targets = dec_targets.view(-1)

    loss = criterion(output, dec_targets)
    loss.backward()
    optimizer.step()

    preds = output.argmax(dim=-1)
    train_loss += loss.detach().cpu().item()
    mask = dec_targets != 0
    correct = (preds == dec_targets) & mask
    train_correct += correct.sum().detach().cpu().item()
    train_tokens += mask.sum().detach().cpu().item()

  train_loss /= len(train_loader)
  train_acc = train_correct / train_tokens
  train_losses.append(train_loss)
  train_accs.append(train_acc)

  model.eval()
  with torch.no_grad():
    val_loss, val_correct, val_tokens = 0, 0, 0

    for enc_inputs, dec_inputs, dec_targets in val_loader:
      enc_inputs = enc_inputs.to(device)
      dec_inputs = dec_inputs.to(device)
      dec_targets = dec_targets.to(device)

      output = model(enc_inputs, dec_inputs)
      output = output.view(-1, output.size(-1))
      dec_targets = dec_targets.view(-1)

      loss = criterion(output, dec_targets)

      preds = output.argmax(dim=-1)
      val_loss += loss.detach().cpu().item()
      mask = dec_targets != 0
      correct = (preds == dec_targets) & mask
      val_correct += correct.sum().detach().cpu().item()
      val_tokens += mask.sum().detach().cpu().item()

    val_loss /= len(val_loader)
    val_acc = val_correct / val_tokens
    val_losses.append(val_loss)
    val_accs.append(val_acc)

  print(f'Epoch {epoch+1}/{epochs} TrainLoss={train_loss:.4f} TrainAcc={train_acc:.4f} ValLoss={val_loss:.4f} ValAcc={val_acc:.4f}')

In [ ]:
torch.save(model, 'seq2seq_chatbot.pth')

---

### 5.만든 모델로 추론

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def translate(input_seq, model, kor_tokenizer, max_len=ans_max_len, device=device):
  model = model.to(device)
  model.eval()
  encoder = model.encoder
  decoder = model.decoder

  input_seq = torch.tensor(input_seq, dtype=torch.long).to(device)

  # Encoder 처리
  with torch.no_grad():
    hidden, cell = encoder(input_seq)

  # Decoder 출력(Auto Regressive)
  sos_index = kor_tokenizer.word_index['<sos>']
  eos_index = kor_tokenizer.word_index['<eos>']

  output_sentences = []

  target_seq = torch.tensor([[sos_index]], dtype=torch.long).to(device)

  with torch.no_grad():
    for _ in range(max_len):
      output, hidden, cell = decoder(target_seq, hidden, cell)  # (batch_size, seq_len, vocab_size)
      proba = output.squeeze(1).softmax(dim=-1)                 # (batch_size, vocab_size)
      pred = proba.argmax(dim=-1).detach().cpu().item()

      if pred == eos_index:
        break

      if pred > 0:
        word = kor_tokenizer.index_word[pred]
        output_sentences.append(word)

      target_seq = torch.tensor([[pred]], dtype=torch.long).to(device)

  return ' '.join(output_sentences)

In [ ]:
import numpy as np

for _ in range(5):
  index = np.random.choice(len(q_inputs))
  input_seq = encoder_inputs[index:index+1]
  output = translate(input_seq, model, kor_tokenizer)

  print(f'Q: {q_inputs[index]}')
  print(f'A: {ans_inputs[index]}')
  print(f'모델의 결과값: {output}')
  print()